In [ ]:
#export 

import os 
import yt_dlp
import logging
import logs
from pathlib import Path
from typing import Optional, Tuple

from datetime import datetime, timedelta
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

YOUTUBE_API_KEY=None

tag="youtuber_manager"

In [2]:
#export 

def build_format_selector(resolution) -> str:
    """
    Devuelve el 'format' selector para yt-dlp según la resolución pedida o 'best'.
    - 'best'  -> máxima calidad, intentando MP4/M4A.
    - int/str -> intenta exactamente esa altura; si no existe, la mejor <= esa altura.
                 Siempre prioriza contenedores MP4 (video) y M4A (audio).

    Acompañar con: ydl_opts['merge_output_format'] = 'mp4'
    """
    # Normalizar resolución
    if resolution is None:
        res = "best"
    elif isinstance(resolution, str):
        r = resolution.strip().lower()
        if r == "best":
            res = "best"
        else:
            # admitir "720p" o "720"
            if r.endswith("p"):
                r = r[:-1]
            res = int(r)
    elif isinstance(resolution, int):
        res = resolution
    else:
        # fallback seguro
        res = "best"

    if res == "best":
        # Máxima calidad, priorizando MP4/M4A
        return (
            "bestvideo[ext=mp4]+bestaudio[ext=m4a]/"
            "best[ext=mp4]/"
            "best"
        )

    height = res
    # 1) Exacta altura pedida (video+audio separados, luego mux a mp4)
    # 2) Exacta altura pedida en single-file MP4
    # 3) Mejor <= altura (dash) en MP4+M4A
    # 4) Mejor <= altura en single-file MP4
    # 5) Último fallback a best
    return (
        f"bestvideo[ext=mp4][height={height}]+bestaudio[ext=m4a]/"
        f"best[ext=mp4][height={height}]/"
        f"bestvideo[ext=mp4][height<={height}]+bestaudio[ext=m4a]/"
        f"best[ext=mp4][height<={height}]/"
        "best"
    )

def get_youtube_resolutions(youtube_url: str) -> list[dict]:
    """
    Devuelve una lista con las resoluciones disponibles para un video de YouTube.

    Parameters
    ----------
    youtube_url : str
        URL del video de YouTube.

    Returns
    -------
    list[dict]
        Lista de diccionarios con información de cada formato de video:
        [
            {'format_id': '18', 'ext': 'mp4', 'height': 360, 'fps': 30, 'vcodec': 'avc1.42001E', 'acodec': 'mp4a.40.2'},
            ...
        ]
    """
    ydl_opts = {'quiet': True, 'noprogress': True}
    results = []

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)
        for f in info.get("formats", []):
            height = f.get("height")
            ext = f.get("ext")
            fid = f.get("format_id")

            # Solo incluimos formatos de video con altura definida
            if height and ext and fid:
                results.append(f"{fid} {ext} {height}p")

    # Ordenamos por altura descendente
    results.sort(key=lambda x: int(x.split()[2][:-1]), reverse=True)
    return results

In [3]:
#export

def download_video_youtube(
    youtube_url: str,
    output_path: str = "Downloads",
    log: int = logging.INFO,
    tag: str = "YT",
    resolution: Optional[str] = None,
    cookies_file: Optional[str] = None,
    cookies_from_browser: Optional[str] = None,  # "chrome", "firefox", "safari", etc.
) -> Tuple[str, str]:
    """
    Descarga un vídeo de YouTube en MP4.

    Nota:
    - Si el vídeo requiere sesión, edad, región o anti-bot, suele funcionar mejor
      usar cookies del navegador o un cookies.txt válido.
    """

    method = "download_video_youtube"

    logger = logging.getLogger(method)
    logger.setLevel(log)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setLevel(log)
        formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    logger.info(f"[{tag}]-{method}-START URL={youtube_url}")

    Path(output_path).mkdir(parents=True, exist_ok=True)

    # Selector de formato más tolerante
    if resolution:
        # Intenta mejor vídeo <= resolución + mejor audio; si no, mp4 combinado; si no, cualquier mejor
        format_selector = (
            f"bestvideo[height<={resolution}]+bestaudio/"
            f"best[height<={resolution}][ext=mp4]/best"
        )
    else:
        format_selector = "bestvideo+bestaudio/best[ext=mp4]/best"

    ydl_opts = {
        "format": format_selector,
        "outtmpl": os.path.join(output_path, "%(id)s.%(ext)s"),
        "merge_output_format": "mp4",
        "retries": 10,
        "fragment_retries": 10,
        "concurrent_fragment_downloads": 3,
        "noprogress": True,
        "quiet": log > logging.DEBUG,
        "verbose": log <= logging.DEBUG,  # clave para diagnosticar
        "source_address": "0.0.0.0",
        "http_headers": {
            "User-Agent": "Mozilla/5.0",
            "Accept-Language": "es-ES,es;q=0.9",
        },
        # Evita algunos problemas de extracción con ciertos clientes
        "extractor_args": {
            "youtube": {
                "player_client": ["android", "web"]
            }
        },
        # A veces ayuda a no reventar por un formato concreto
        "ignoreerrors": False,
    }

    # Cookies: usa UNA estrategia, no varias a la vez salvo que sepas por qué
    if cookies_from_browser:
        # En Python API suele usarse esta clave con tupla/lista según entorno/versiones;
        # esta forma es la más habitual.
        ydl_opts["cookiesfrombrowser"] = (cookies_from_browser,)
        logger.info(f"[{tag}]-{method}-Usando cookies del navegador: {cookies_from_browser}")
    elif cookies_file:
        if not os.path.exists(cookies_file):
            raise FileNotFoundError(f"No existe el fichero de cookies: {cookies_file}")
        ydl_opts["cookiefile"] = cookies_file
        logger.info(f"[{tag}]-{method}-Usando cookiefile: {cookies_file}")

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(youtube_url, download=False)

            if info is None:
                raise RuntimeError("yt-dlp no devolvió metadatos para la URL")

            video_id = info["id"]
            ext = info.get("ext", "mp4")
            file_name = f"{video_id}.{ext}"
            file_path = os.path.join(output_path, file_name)

            logger.info(f"[{tag}]-{method}-ID={video_id} Título={info.get('title')}")

            if not os.path.exists(file_path):
                ydl.download([youtube_url])

                # Tras merge/remux, a veces la extensión final termina siendo mp4
                possible_mp4 = os.path.join(output_path, f"{video_id}.mp4")
                if os.path.exists(possible_mp4):
                    file_name = f"{video_id}.mp4"
                    file_path = possible_mp4
            else:
                logger.info(f"[{tag}]-{method}-Ya existe: {file_path}")

        logger.info(f"[{tag}]-{method}-FIN")
        return file_name, os.path.abspath(file_path)

    except yt_dlp.utils.DownloadError as e:
        logger.exception(f"[{tag}]-{method}-DownloadError: {e}")
        raise RuntimeError(
            "yt-dlp no pudo extraer/descargar el vídeo. "
            "Suele deberse a cookies inválidas, rate limiting, restricción regional/edad, "
            "o cambios recientes de YouTube."
        ) from e
    
'''    
def download_video_youtube(
    youtube_url: str,
    output_path: str = "Downloads",
    log: int = logging.INFO,
    tag: str = "YT",
    resolution: str = None 
) -> tuple[str, str]:
    """
    Descarga un video de YouTube en formato MP4.

    Parameters
    ----------
    youtube_url : str
        URL de YouTube del video que quieres descargar.
    output_path : str, optional
        Carpeta donde se guardará el archivo. Por defecto es "Downloads".
    log : int, optional
        Nivel de logging (logging.DEBUG, logging.INFO, etc.). Por defecto es INFO.
    tag : str, optional
        Etiqueta que se mostrará en los mensajes de log. Por defecto es "YT".

    Returns
    -------
    file : str
        Nombre del fichero descargado (ej. "dQw4w9WgXcQ.mp4").
    file_path : str
        Ruta absoluta donde se guardó el fichero.
    """
    method = "download_video_youtube"

    # ---- 1. Configuración del logger  ------------------------------------
    logger, handler = logs.crea_log(log, method)
    logger.info(f"[{tag}]-{method}-START Descargando video: {youtube_url}")

    # ---- 2. Aseguramos que exista la carpeta de salida  ------------------
    if not os.path.exists(output_path):
        os.makedirs(output_path)
        logger.debug(f"[{tag}]-{method}-Creado directorio '{output_path}'")

    # Si el nivel es INFO queremos suprimir los mensajes de yt-dlp
    quiet = log == logging.INFO

    # Descargamos en distintas resoluciones
    format_selector = build_format_selector(resolution)

    # ---- 3. Opciones de yt_dlp  ------------------------------------------
    ydl_opts = {
        # Mejor calidad de video y audio, con salida en MP4
        'format': format_selector,
        'outtmpl': f'{output_path}/%(id)s.%(ext)s',
        'quiet': True,          # suprime mensajes de yt-dlp
        "retries": 10,
        "fragment_retries": 10,
        "concurrent_fragment_downloads": 5,
        # Forzar IPv4 (evita ciertos 403)
        "source_address": "0.0.0.0",
        # Pasar cookies del navegador (age/geo/privado). Cambia 'chrome' si usas otro.
        "cookies": "cookies.txt",   
        # Cabeceras modernas (a veces ayuda)
        "http_headers": {
            "User-Agent": "Mozilla/5.0",
            "Accept-Language": "es-ES,es;q=0.9"
        },
        'noprogress': True,     # no muestra barra de progreso
        'logger': logger,       # usamos nuestro logger
    }
    logger.debug(f"[{tag}]-{method}-Opciones yt_dlp configuradas.")

    # ---- 4. Descarga (o verificación de existencia)  --------------------
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        # 4a. Obtener info sin descargar (para saber el id)
        info = ydl.extract_info(youtube_url, download=False)

        file = f"{info['id']}.mp4"
        file_path = os.path.join(output_path, file)

        if not os.path.exists(file_path):
            # 4b. Descargamos realmente
            logger.debug(f"[{tag}]-{method}-Descargando a {file_path}")
            ydl.download([youtube_url])          # descarga con las opciones ya configuradas
            logger.debug(f"[{tag}]-{method}-Archivo descargado: {file_path}")
        else:
            logger.debug(f"[{tag}]-{method}-El archivo ya existe: {file_path}")

    # ---- 5. Cerramos logger y devolvemos resultados --------------------
    logger.info(f"[{tag}]-{method}-FIN")
    logs.cierra_log(logger, handler)

    return file, file_path
'''


'    \ndef download_video_youtube(\n    youtube_url: str,\n    output_path: str = "Downloads",\n    log: int = logging.INFO,\n    tag: str = "YT",\n    resolution: str = None \n) -> tuple[str, str]:\n    """\n    Descarga un video de YouTube en formato MP4.\n\n    Parameters\n    ----------\n    youtube_url : str\n        URL de YouTube del video que quieres descargar.\n    output_path : str, optional\n        Carpeta donde se guardará el archivo. Por defecto es "Downloads".\n    log : int, optional\n        Nivel de logging (logging.DEBUG, logging.INFO, etc.). Por defecto es INFO.\n    tag : str, optional\n        Etiqueta que se mostrará en los mensajes de log. Por defecto es "YT".\n\n    Returns\n    -------\n    file : str\n        Nombre del fichero descargado (ej. "dQw4w9WgXcQ.mp4").\n    file_path : str\n        Ruta absoluta donde se guardó el fichero.\n    """\n    method = "download_video_youtube"\n\n    # ---- 1. Configuración del logger  ------------------------------------

In [4]:
#export 

def download_audio_youtube(youtube_url, output_path="Downloads", log=logging.INFO): 
    """
    La función `download_audio_youtube` descarga un audio de YouTube a partir de la url que 
    recibe. 

    Parameters:
    - youtube_url (str): La URL de YouTube del video del que hay que descargar el audio. 
    - output_path (str): La ruta donde se guard el audio descargado. Por defecto, Downloads
    - log (logging): El nivel de registro para el logger

    Exception:
        None.

    Returns:
        Devuelve el parámetro file, con el nombre del fichero, y file_path con el path completo incluyendo 
        el nombre del fichero. 
    """
    method="download_audio_youtube"

    logger, handler = logs.crea_log(log, method)
    logger.info(f"[{tag}]-{method}-START Iniciando descarga audio para la url {youtube_url}")

    if not os.path.exists(output_path):
        os.makedirs(output_path)
        logging.debug(f"[{tag}]-{method}- Creado directorio '{output_path}'")

    #Si se llama con modo Info no queremos que se muestre la información. 
    quiet = log == logging.INFO 

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'{output_path}/%(id)s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '128',
        }],
        'quiet': True,  # Suprime los mensajes de yt-dlp
        'noprogress': True,
        'logger': logger 
    }
    logger.debug(f"{method}- Youtube options configured.")
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        #Recuperamos la información sin descargar. 
        info = ydl.extract_info(youtube_url, download=False)

        #Montamos el nombre del fichero. 
        file = f"{info['id']}.mp3"
        file_path = f"{output_path}/{file}"

        #Verificamos si ya existe el fichero o no.
        if not os.path.exists(file_path):
            #Descargamos la información. 
            info = ydl.extract_info(youtube_url, download=True)
            logger.debug(f"[{tag}]-{method}- fichero descargado {file_path}")
        else:
            logger.debug(f"[{tag}]-{method}- fichero ya existe -> {file_path}")
        
        
        logger.debug(f"[{tag}]-{method}-END")
        logs.cierra_log(logger, handler)

        return file, file_path

In [5]:
#export 

def get_video_duration(youtube_url, unidad="min", level_log = logging.INFO):
    method="get_video_duration"

    logger, handler = logs.crea_log(level_log, method)
    logger.info(f"{tag}-{method}-START Comprobando duración {youtube_url}")

    ydl_opts = {
        'quiet': True,  # Suprime los mensajes de yt-dlp
        'noprogress': True,
        'skip_download': True 
    }
    logger.debug(f"[{tag}]-{method}- Youtube options configured.")

    duration = -1
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)
        duration=info.get("duration", 0)  #Devuelve la duración en segundos. 

        if unidad == "min":
            duration = duration/60

    logs.cierra_log(logger, handler)

    return duration

In [6]:
#export

def get_video_visits(
    youtube_url: str,
    level_log: int = logging.INFO,
    tag: str = "YT",
    cookies_file: str | None = None,
    cookies_from_browser: str | None = None,
) -> int:
    method = "get_video_visits"
    num_visits = -1

    logger, handler = logs.crea_log(level_log, method)
    logger.info(f"[{tag}]-{method}-START Comprobando visitas: {youtube_url}")

    ydl_opts = {
        "quiet": level_log > logging.DEBUG,
        "verbose": level_log <= logging.DEBUG,
        "skip_download": True,
        "noplaylist": True,
        "extract_flat": False,
        "http_headers": {
            "User-Agent": "Mozilla/5.0",
            "Accept-Language": "es-ES,es;q=0.9",
        },
    }

    if cookies_from_browser:
        ydl_opts["cookiesfrombrowser"] = (cookies_from_browser,)
    elif cookies_file:
        ydl_opts["cookiefile"] = cookies_file

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info_dict = ydl.extract_info(youtube_url, download=False)

            if info_dict and info_dict.get("view_count") is not None:
                num_visits = int(info_dict["view_count"])

        logger.debug(f"[{tag}]-{method}-END num_visits={num_visits}")
        return num_visits

    except yt_dlp.utils.DownloadError as e:
        logger.exception(f"[{tag}]-{method}-ERROR {e}")
        return -1

    finally:
        logs.cierra_log(logger, handler)

In [7]:
#export

def get_list_videos(channel_id, fromDays, level_log=logging.INFO):
    """
    Downloads videos published in the last 'fromDays' days from a specified YouTube channel.

    Args:
        channel_id (str): The ID of the YouTube channel.
        fromDays (int, optional): Number of days to look back for published videos. Defaults to 7.
        level_log (logging.Level, optional): Logging level. Defaults to logging.INFO.

    Returns:
        list: A list of video IDs and titles that match the criteria.
    """
    method="get_list_videos"

    # Configure logging
    logger, handler = logs.crea_log(level_log, method)
    logger.debug(f"{tag}-{method}-START channel_id={channel_id} en los últimos {fromDays} días")

    # Calculate the date to look back from
    #today = datetime.now()
    #lookback_date = today - timedelta(days=fromDays)
    lookback_date=fromDays
    #lookback_date_str = lookback_date.strftime("%Y-%m-%dT%H:%M:%SZ")

    # Initialize YouTube API client 
    youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

    try:
        # Get the uploads playlist ID for the channel
        request = youtube.channels().list(
            part='contentDetails',
            id=channel_id
        )
        response = request.execute()
        uploads_playlist_id = response['items'][0]['contentDetails']['relatedPlaylists']['uploads']

        # Get video IDs from the uploads playlist
        videos = []
        next_page_token = None

        while True:
            request = youtube.playlistItems().list(
                part='snippet',
                playlistId=uploads_playlist_id,
                maxResults=50,
                pageToken=next_page_token
            )
            response = request.execute()

            for item in response['items']:
                video_date_str = item['snippet']['publishedAt']
                video_date = datetime.strptime(video_date_str, '%Y-%m-%dT%H:%M:%SZ')
                if video_date.timestamp() >= lookback_date.timestamp():
                    videos.append({
                        'video_id': item['snippet']['resourceId']['videoId'],
                        'title': item['snippet']['title'],
                        'date': video_date.strftime('%Y-%m-%d')
                    })

            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                break

        logger.debug(f"[{tag}]-{method}- Encontrados {len(videos)} videos publicados.")

        return videos

    except HttpError as e:
        logger.error(f"An error occurred: {e}")
        raise
    except KeyError as e:
        logger.error(f"[{tag}]-{method}-ERROR Unexpected response format from YouTube API.")
        raise
    finally: 
        logs.cierra_log(logger, handler)

    

In [8]:
#################################################
#################################################
#################################################

In [ ]:
ydl_opts = {
        'quiet': True,  # Suprime los mensajes de yt-dlp
        'noprogress': True,
        'skip_download': True 
    }
    
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info("https://www.youtube.com/watch?v=RyDK3BlmLLo&pp=ygUSam9zZXRlIGVsIGNhbmlsbGFz", download=False)
info.get("channel_id", 0)

In [ ]:
# Example usage
channel_id = 'UCBAOfVo-X3iRZWVzFyL_X8g'  # Replace with the actual channel ID
videos = get_list_videos(channel_id, fromDays=7)
sorted_videos = sorted(videos, key=lambda x: x['date'])
for video in sorted_videos:
    print(f"Created: {video['date']}, Title: {video['title']}, url: https://www.youtube.com/watch?v={video['video_id']}")

In [ ]:
#################################################
#################################################

In [9]:
youtube_url="https://youtu.be/f7cFhawgodA?si=GTbIoW-HBfH6LJN1"
#logging.handlers.clear()

#video 2 --> https://youtu.be/mO8lkgdNMu8?si=pp-wkfzABjf58xqF
#video 3 --> https://youtu.be/LUjvxlnl91k?si=d_pZZqIa6rxLdY9_

usuario = 'Users/zzddfge'
carpeta = 'Downloads'
disco_duro = '/Volumes/Macintosh HD'

output_path=os.path.join(disco_duro, usuario, carpeta)
download_audio_youtube(youtube_url, output_path=output_path, log=logging.INFO)
#download_video_youtube(youtube_url, output_path=output_path, log=logging.INFO)



2026-05-27 21:43:38,229 - INFO - [youtuber_manager]-download_audio_youtube-START Iniciando descarga audio para la url https://youtu.be/f7cFhawgodA?si=GTbIoW-HBfH6LJN1
2026-05-27 21:43:39,135 - WARNING - [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be missing. See  https://github.com/yt-dlp/yt-dlp/wiki/EJS  for details on installing one
2026-05-27 21:43:40,120 - WARNING - [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be missing. See  https://github.com/yt-dlp/yt-dlp/wiki/EJS  for details on installing one


('f7cFhawgodA.mp3',
 '/Volumes/Macintosh HD/Users/zzddfge/Downloads/f7cFhawgodA.mp3')

In [8]:
!pip install -U yt-dlp 


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 4.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: yt-dlp
    Found existing installation: yt-dlp 2025.10.22
    Uninstalling yt-dlp-2025.10.22:
      Successfully uninstalled yt-dlp-2025.10.22

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [10]:
youtube_url="https://youtu.be/wuog9_p_ycc?si=AQugVjZAOPQggfxY"
#logging.handlers.clear()

usuario = 'Users/zzddfge'
carpeta = 'Downloads'
disco_duro = '/Volumes/Macintosh HD'

output_path=os.path.join(disco_duro, usuario, carpeta)
get_youtube_resolutions(youtube_url)
download_video_youtube(youtube_url, output_path=output_path, log=logging.INFO,resolution="480")



         player = https://www.youtube.com/s/player/b32979e9/player_ias.vflset/en_US/base.js
         n = 5P1OXEf1Dns4kmOl ; player = https://www.youtube.com/s/player/b32979e9/player_ias.vflset/en_US/base.js
         Please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U
2025-11-18 22:28:58,165 - INFO - [YT]-download_video_youtube-START Descargando video: https://youtu.be/wuog9_p_ycc?si=AQugVjZAOPQggfxY
2025-11-18 22:28:59,899 - WARNING - [youtube] Falling back to generic n function search
         player = https://www.youtube.com/s/player/b32979e9/player_ias.vflset/en_US/base.js
2025-11-18 22:28:59,908 - WARNING - [youtube] wuog9_p_ycc: nsig extraction failed: Some formats may be missing
         n = -nZev2Keb9QdJkEr ; player = https://www.youtube.com/s/player/b32979e9/player_ias.vflset/en_US/base.js
         Please report this issue on  https://github.com/yt-dlp/yt-dlp

('wuog9_p_ycc.mp4',
 '/Volumes/Macintosh HD/Users/zzddfge/Downloads/wuog9_p_ycc.mp4')

In [ ]:
import yt_dlp

def test_basic_download(youtube_url):
    ydl_opts = {
        'format': 'bestaudio',
        'outtmpl': '%(title)s.%(ext)s',
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])

test_basic_download("https://youtu.be/JiMJp2_wrqM?si=_5lTZqLWtEGbWeFb")

In [11]:
youtube_url="https://www.youtube.com/watch?v=LUjvxlnl91k"

get_video_visits(youtube_url=youtube_url)

2025-11-08 18:38:01,605 - INFO - youtuber_manager-get_video_vistis-START Comprobando duración https://www.youtube.com/watch?v=LUjvxlnl91k
2025-11-08 18:38:01,605 - INFO - youtuber_manager-get_video_vistis-START Comprobando duración https://www.youtube.com/watch?v=LUjvxlnl91k


205